Let's Check CICD -- part 3

##### **_IMPORT QUERIES_**

In [0]:
from pyspark.sql.functions import col, expr, lit, when, regexp_replace, count, desc, rank, to_date, substring, countDistinct, sum, to_timestamp, to_date, min, max,datediff, year, month, broadcast,avg, round

In [0]:
from pyspark.sql.types import StructField, StructType, IntegerType, StringType, DateType, FloatType, DoubleType

_Histogram of Tweets_

In [0]:
# Define schema
s = StructType([
  StructField("tweet_id", IntegerType(), True),
  StructField("user_id", IntegerType(), True),
  StructField("msg", StringType(), True),
  StructField("tweet_date", StringType(), True)
])
# Define data
d = [
    [241425, 254, "If the salary is so competitive why won’t you tell me what it is?", "3/1/2022"],
    [214252, 111, "Am considering taking Tesla private at $420. Funding secured.", "12/30/2021"],
    [739252, 111, "Despite the constant negative press covfefe", "1/1/2022"],
    [846402, 111, "Following @NickSinghTech on Twitter changed my life!", "2/14/2022"],
    [231574, 148, "I no longer have a manager. I can't be managed", "3/23/2022"]
]

df = spark.createDataFrame(d, s)

# Filter rows where the year of tweet_date is 2022
df_filtered = df.filter(substring(col("tweet_date"), -4, 4) == "2022")

# First aggregation: group by user_id and count distinct tweet_id, then order by tweet_count
res = df_filtered.groupBy("user_id") \
    .agg(
        countDistinct("tweet_id").alias("tweet_count")
    ) \
    .orderBy(col("tweet_count").asc())

# Show the intermediate result
res.show(truncate=False)

# Second aggregation: group by tweet_count and count how many users have that tweet_count
df_final = res.groupBy("tweet_count").agg(count("user_id").alias("total_user_count"))

# Show the final result
df_final.show(truncate=False)

_Data Science Skills_

In [0]:
s = StructType([
  StructField("skill", StringType(), True),
  StructField("c_id", IntegerType(), True),
])
# Define data
d = [
["Python",	123],
["Tableau",	123],
["PostgreSQL",	123],
["R",	234],
["PowerBI",	234],
["SQL Server",	234],
["Python",	345],
["Tableau",	345],

]

df = spark.createDataFrame(d, s)

df.show(truncate = False)

df_fil = df.filter(col("skill").isin("Python", "Tableau", "PostgreSQL"))
df_fil.show(truncate = False)

df_fil = df_fil.groupBy("c_id").agg(count("skill").alias("abv")) \
    .filter(col("abv") == 3) \
    .select(col("c_id").alias("candidate_id")) \
    .show()


_Page With No Likes_

In [0]:
s = StructType([
  StructField("page_id", IntegerType(), True),
  StructField("page_name", StringType(), True),
])

# Define schema for the second DataFrame with DateType
s2 = StructType([
  StructField("user_id", IntegerType(), True),
  StructField("page_id", IntegerType(), True),
  StructField("liked_date", StringType(), True),  # DateType field
])

# Define data for the first DataFrame
d = [
    [20001, "SQL Solutions"],
    [20045, "Brain Exercises"],
    [20701, "Tips for Data Analysts"],
    [31111, "Postgres Crash Course"],
    [32728, "Break the thread"]
]

# Define data for the second DataFrame with dates as strings in yyyy-MM-dd format
d2 = [
    [111, 20001, "2022-04-08"],
    [121, 20045, "2022-03-12"],
    [156, 20001, "2022-07-25"],
    [255, 20045, "2022-07-19"],
    [125, 20001, "2022-07-19"],
    [144, 31111, "2022-06-21"],
    [125, 31111, "2022-07-04"],
]

# Create DataFrames using the defined schema
df = spark.createDataFrame(d, s)
df2 = spark.createDataFrame(d2, s2)

# Show DataFrames
df.show(truncate=False)

df2 = df2.withColumn("date", to_date(col("liked_date"), "yyyy-mm-dd")).select("user_id", "page_id","date")
df2.show(truncate=False)


df.join(df2, how="left", on=df.page_id == df2.page_id) \
   .select(df.page_id, df2.user_id) \
    .filter(df2.user_id.isNull()) \
      .select(col("page_id")) \
        .orderBy(col("page_id"), ascending = True) \
          .show(truncate=False)




_Unfinished Parts_

In [0]:
_s_up = StructType(
  [
    StructField("part", StringType()),
    StructField("finshed_date", StringType()),
    StructField("assembly_step", IntegerType()),
  ]
)

_d_up = [
  ['battery',	'1/22/2022 0:00',	1],
  ['battery',	'2/22/2022 0:00',	2],
  ['battery',	'3/22/2022 0:00',	3],
  ['bumper',	'1/22/2022 0:00',	1],
  ['bumper',	'2/22/2022 0:00',	2],
  ['bumper',	None,	3],
  ['bumper',	None,	4],
  ['door',	'1/22/2022 0:00',	1],
  ['door',	'2/22/2022 0:00',	2],
  ['engine',	'1/1/2022 0:00',	1],
  ['engine',	'1/25/2022 0:00',	2],
  ['engine',	'2/28/2022 0:00',	3],
  ['engine',	'4/1/2022 0:00',	4],
  ['engine',	None,	5],
]

df_up = spark.createDataFrame(_d_up, _s_up)

# Not in current purview but to Drop Null records .dropna(subset = [] )
df_up_null_drop = df_up.dropna(subset=["finshed_date"]).select(col("part"), col("finshed_date"),col("assembly_step"))

df_up_filter_na = df_up.where(df_up.finshed_date.isNull()).\
                  select(col("part"),col("assembly_step")).\
                  orderBy(df_up.assembly_step.desc()).\
                  display()

_Laptop vs. Mobile Viewership_

In [0]:
_s_lm = StructType([
  StructField("user_id", IntegerType()),
  StructField("device_type", StringType()),
  StructField("view_time", StringType())
])

_d_lm = [
  [123,	'tablet',	'01/02/2022 00:00:00'],
  [125,	'laptop',	'01/07/2022 00:00:00'],
  [128,	'laptop',	'02/09/2022 00:00:00'],
  [129,	'phone',	'02/09/2022 00:00:00'],
  [145,	'tablet',	'02/24/2022 00:00:00']
]

df_lm = spark.createDataFrame(_d_lm,_s_lm)

###example of groupBy
# df_lm.groupBy("device_type").agg(count(df_lm.user_id)).display()

df_lm_final = df_lm.agg(
      sum(when(df_lm.device_type == "laptop", 1).otherwise(0)).alias("l_views"),
      sum(when(df_lm.device_type != "laptop", 1).otherwise(0)).alias("m_views")
      ).display()


###Using Expr
# df_aggregated = df.agg(
#     expr("SUM(CASE WHEN device_type = 'laptop' THEN 1 ELSE 0 END)").alias("laptop_views"),
#     expr("SUM(CASE WHEN device_type IN ('phone', 'tablet') THEN 1 ELSE 0 END)").alias("mobile_views")
# )


_Average Post Hiatus (Part 1)_

In [0]:
_s_aph = StructType(
  [
    StructField("user_id", IntegerType()),
    StructField("post_id", IntegerType()),
    StructField("post_contend", StringType()),
    StructField("Date", StringType())
  ]
) 

_d_aph = [
[151652,	111766,	'its always winter, but never Christmas.' ,	'12/01/2021 11:00:00'],
[661093,	442560,	'Bed. Class 8-12. Work 12-3. Gym 3-5 or 6. Then class 6-10. Another day thats gonna fly',	'09/08/2021 10:00:00'],
[661093,	624356,	'Happy valentines!',	'02/14/2021 00:00:00'],
[151652,	599415,	'Need a hug',	'01/28/2021 00:00:00'],
[178425,	157336,	'Im so done with these restrictions - I want to travel!!!',	'03/24/2021 11:00:00'],
[423967,	784254,	'Just going to cry myself to sleep after watching Marley and Me.',	'05/05/2021 00:00:00'],
[151325,	613451,	'Happy new year all my friends!',	'01/01/2022 11:00:00'],
[151325,	987562,	'The global surface temperature for June 2022 was the sixth-highest in the 143-year recppening.',	'07/01/2022 10:00:00'],
[661093,	674356,	'Cant wait to start my freshman year - super excited!',	'08/18/2021 10:00:00'],
[151325,	451464,	'Garage sale this Saturday 1 PM. All welcome to check out!',	'10/25/2021 10:00:00'],
[151652,	994156,	'Does anyone have an extra iPhone charger to sell?',	'04/01/2021 10:00:00']
]

df_aph = spark.createDataFrame(_d_aph, _s_aph)

df_aph_min_max = df_aph.withColumn("casted_col", to_date(to_timestamp(col("Date"), "MM/dd/yyyy HH:mm:ss"))).\
                        where(year(col("casted_col")) == 2021).\
                        groupBy("user_id").\
                        agg(
                            min("casted_col").alias("min_date"), 
                            max("casted_col").alias("max_date")
                           )
 
df_aph_diff =  df_aph_min_max.withColumn("diff", datediff(col("max_date"), col("min_date"))).\
                              where(col("diff") > 0).select(col("user_id"), col("Diff")).\
                              display()

_Teams Power Users_

In [0]:
_s_tpu = StructType(
  [
    StructField("message_id", IntegerType()), 
    StructField("sender_id", IntegerType()), 
    StructField("reciever_id", IntegerType()), 
    StructField("sent_date", StringType())
  ]
)

_d_tpu = [
[901,	3601,	4500,	'08/03/2022 16:43:00'],
[743,	3601,	8752,	'06/14/2022 14:30:00'],
[888,	3601,	7855,	'08/12/2022 08:45:00'],
[100,	2520,	6987,	'08/16/2021 00:35:00'],
[898,	2520,	9630,	'08/13/2022 14:35:00'],
[990,	2520,	8520,	'08/19/2022 06:30:00'],
[819,	2310,	4500,	'07/10/2022 15:55:00'],
[922,	3601,	4500,	'08/10/2022 17:03:00'],
[942,	2520,	3561,	'08/17/2022 13:44:00'],
[966,	3601,	7852,	'08/17/2022 02:20:00'],
[902,	4500,	3601,	'08/03/2022 06:50:00']
]

df_tpu = spark.createDataFrame(_d_tpu, _s_tpu)

df_tpu.withColumn("casted_col", to_date(to_timestamp(col("sent_date"), "MM/dd/yyyy HH:mm:ss")))\
      .where((month(col("casted_col")) == 8) & (year(col("casted_col")) == 2022))\
      .dropDuplicates(["message_id"])\
      .groupBy(col("sender_id"))\
      .agg(count(col("message_id")).alias("message_count"))\
      .limit(2)\
      .display()




_Duplicate Job Listings_

In [0]:
_s_djl = StructType([
  StructField("company_id", IntegerType()),
  StructField("title", StringType()),
  StructField("job_id", IntegerType()),
  StructField("desc", StringType()),
])

_d_djl = [
[827,	'Business Analyst',	248,	'Business analyst evaluates past and current business data with the primary goal of improving decision-making processes within organizations.'],

[845,	'Business Analyst',	149,	'Business analyst evaluates past and current business data with the primary goal of improving decision-making processes within organizations.'],

[345,	'Data Analyst',	945,	'Data analyst reviews data to identify key insights into a business\'s customers and ways the data can be used to solve problems.'],
[345,	'Data Analyst',	164,	'Data analyst reviews data to identify key insights into a business\'s customers and ways the data can be used to solve problems.'],
[244,	'Data Engineer',	172,	'Data engineer works in a variety of settings to build systems that collect, manage, and convert raw data into usable information for data scientists and business analysts to interpret.'],
[827,	'Data Scientist',	256,	'Data scientist uses data to understand and explain the phenomena around them, and help organizations make better decisions.'],
[244,	'Software Engineer',	365,	'Software engineers design and create computer systems and applications to solve real-world problems.'],
[400,	'Business Intelligence Analyst',	674,	'Business intelligence analyst reviews data to produce finance and market intelligence reports.'],
[827,	'Data Scientist',	245	,'Data scientist uses data to understand and explain the phenomena around them, and help organizations make better decisions.'],
[244,	'Software Engineer',	301,	'Software engineers design and create computer systems and applications to solve real-world problems.']
]

df_djl = spark.createDataFrame(_d_djl,_s_djl)

df_djl_final = df_djl\
  .groupBy(col("company_id"))\
  .agg(countDistinct(col("desc")).alias("cnt"))\
  .where(col("cnt") == 1 )\
  .agg(sum(col("cnt")).alias("total_duplicate_companies")) 
  
df_djl_final.display(truncate = "false")


# a = df_djl.groupBy('company_id')\
#       .agg(
#         countDistinct(col('job_id')).alias('a')
#       )\
#       .filter(col('a') > 1)
# a.count()

_Cities With Completed Trades_

In [0]:
_s_cct_t = StructType(
  [
    StructField("user_id", IntegerType()),
    StructField("status", StringType())
  ]
)

_s_cct_u = StructType(
  [
    StructField("user_id", IntegerType()),
    StructField("city", StringType())
  ]
)

_d_cct_t = [
  [111,	'Cancelled'],
  [111,	'Completed'],
  [148,	'Completed'],
  [300,	'Completed'],
  [488,	'Completed'],
  [148,	'Completed'],
  [148,	'Completed'],
  [265,	'Completed'],
  [488,	'Cancelled'],
  [265,	'Completed'],
  [178,	'Completed'],
  [178,	'Completed']
]

_d_cct_u = [
[111,	'San Francisco'],
[148,	'Boston'],
[178,	'San Francisco'],
[265,	'Denver'],
[300,	'San Francisco'],
[488,	'New York']
]

df_cct_t = spark.createDataFrame(_d_cct_t, _s_cct_t)
df_cct_u = spark.createDataFrame(_d_cct_u, _s_cct_u)

df_cct_joined_filtered = df_cct_t\
              .join(
                broadcast(df_cct_u), #BROADCAST JOIN
                on = df_cct_t.user_id == df_cct_u.user_id, 
                how = "left")\
              .where(col("status") == "Completed" )\
              .select(df_cct_t["user_id"], col("city"))
           

df_cct_final = df_cct_joined_filtered\
               .groupBy(col("city"))\
               .agg(count(col("user_id")).alias("total_orders"))\
               .orderBy(col("total_orders").desc())\
               .limit(3)\
               .display()

# Solution 2
# df_cct_t.where(col('status') == 'Completed')\
#         .join(df_cct_u, how = 'left', on = df_cct_t.user_id == df_cct_u.user_id)\
#         .groupBy(col('city'))\
#         .agg(
#           count(df_cct_t["user_id"]).alias("distinct_user_count"))\
#             .orderBy(col('distinct_user_count').desc()).limit(3).display()

_Average Review Ratings_

In [0]:
_acr_schema = StructType([
  StructField("product_id", IntegerType()),
  StructField("stars", IntegerType()),
  StructField("date", StringType())
])

_acr_data = [
  [5001, 4, '08/16/2022 12:00:00'],
  [69852,	4,	'10/28/2022 12:00:00'],
  [50001,	3,	'10/04/2021 12:00:00'],
  [69852,	3,	'10/06/2024 12:00:00'],
  [69852,	2,	'09/16/2024 12:00:00']
]

df_acr = spark.createDataFrame(_acr_data, _acr_schema)


# df_acr_1 = df_acr.withColumn("casted_col", to_date(to_timestamp(col("date"), "MM/dd/yyyy HH:mm:ss")))\
#       .withColumn("month_num", month(col("casted_col")))\
#       .groupBy(col("product_id"), col("month_num"))\
#       .agg(round(avg(col("stars")),2).alias("s_avg"))\
#       .orderBy(col("month_num"), col("product_id").desc())

# df_acr_1.select(
#         col("month_num").alias("mnth"),
#         col("product_id").alias("product_id"),
#         col("s_avg").alias("avg_stars"),
#         ).display()

df_acr.withColumn('c_date', to_date(to_timestamp(col('date'), "MM/dd/yyyy HH:mm:ss")))\
      .withColumn('mth', month(col('c_date')))\
      .groupBy(col('mth'), col('product_id'))\
      .agg(avg(col('stars')).alias('avg_stars'))\
      .orderBy(col('mth'), col('product_id'))\
      .display()


_App Click-through Rate (CTR)_

In [0]:
_ctr1_schema = StructType([
  StructField("id",IntegerType()),
  StructField("event",StringType()),
  StructField("date",StringType()),
])

_ctr1_data = [
[123,	'impression',	'07/18/2022 11:36:12'],
[123,	'impression',	'07/18/2022 11:37:12'],
[123,	'click',	'07/18/2022 11:37:42'],
[234,	'impression',	'08/18/2022 14:15:12'],
[234,	'click',	'08/18/2022 14:16:12'],
[123,	'impression',	'10/23/2021 12:11:56'],
[123,	'click',	'10/23/2021 12:22:12'],
[123,	'impression',	'04/06/2022 13:13:13'],
[123,	'click',	'04/07/2022 12:20:30'],
[234,	'impression',	'02/09/2022 10:05:02'],
[234,	'impression',	'05/20/2022 12:00:00']
]

df_ctr1_1 = spark.createDataFrame(_ctr1_data, _ctr1_schema)

df_ctr1_2 = df_ctr1_1.withColumn("date_final", to_date(to_timestamp(col("date"), "MM/dd/yyyy HH:mm:ss")))\
                     .where(year(col("date_final")) == 2022)\
                     .groupBy(col("id"), col("event"))\
                     .agg(
                       sum(when(col("event") == "impression", 1).otherwise(0)).alias("imp_cnt"),
                       sum(when(col("event") == "click", 1).otherwise(0)).alias("click_cnt"),
                     )\
                     .select(col("id"), col("imp_cnt"), col("click_cnt"))\
                   

df_ctr1_final = df_ctr1_2.groupBy(col("id"))\
                         .agg(
                           round(((sum(col("click_cnt")) / sum(col("imp_cnt")))*100),2).alias("ctr") 
                         )\
                        .orderBy(col("id"))\
                        .display()


_Pharmacy Analytics (Part 1)_ Casting before creating dataframe

In [0]:
_pa_1_schema = StructType([
  StructField("drugs", StringType()),
  StructField("cogs", DoubleType()),
  StructField("total_sales", DoubleType()),
])

_pa_data = [
['Active-Medicated specimen collection', 1124982.66, 1460189.98],
['Acyclovir', 3427421.73, 3130097],
['Acyclovir Sodium', 616632.58, 841911.12],
['Alimta', 575583.41, 1194250.01],
['Allopurinol', 710573.44, 1425027.99],
['Alprazolam', 2014858.59, 3080474.64],
['Amitriptyline Hydrochloride', 2705112.11, 4701990.22],
['Amoxicillin and Clavulanate Potassium', 981993.64, 1518343],
['ANASTROZOLE', 2560951.38, 3116140.22],
['Androgel', 675055.5, 3025655],
['Antiseptic Hand Gel', 436399.12, 1734572],
['Armour Thyroid', 956693.99, 1151498.6],
['AVAPRO', 2327541.25, 2792775.1],
['BANANA BOAT SUNSCREEN', 163610.19, 937901.5],
['Brucella Remedy', 221803.73, 700562.13],
['Budpak Petroleum Jelly', 107902.51, 1406533.69],
['Burkhart', 1006447.73, 1084258],
['Cefaclor', 908662.72, 1793428.19],
['Childrens Ibuprofen', 384241.13, 2835232.18],
['Chlorzoxazone', 595485.21, 1336647.6],
['Cialis', 3189863.82, 9654492.33],
['Ciprofloxacin', 1278017.17, 2115658],
['Citalopram', 407210.29, 1201626.71],
['Citalopram Hydrobromide', 412421.37, 1355861.1],
['Clarithromycin', 3692136.66, 3499574.92],
['Claritin', 412654.26, 2825302],
['clobetasol propionate', 1281909.78, 2582356],
['Clobetasol Propionate', 1754344.77, 3303186.78],
['Clotrimazole', 2707620.02, 2717420.96],
['Cloves', 1202177.41, 1892705.42],
['Common Sagebrush', 2755697.86, 3059122.36],
['Crab', 1164637.24, 1698544.77],
['Cuvposa', 934986.87, 2933877],
['Dermasorb TA Complete Kit', 2742445.9, 2521023.73],
['DHEA', 1233306.52, 1559816],
['Diaper Rash Skin Protectant Crema Cer', 426539.59, 537795],
['Diclofenac Sodium', 482212.61, 1545347.97],
['Diltiazem Hydrochloride', 1041602.54, 1796908.48],
['Diphenhydramine HCL', 256621.14, 778332.7],
['DIPHENHYDRAMINE HYDROCHLORIDE', 453989.65, 1825783.98],
['Divalproex sodium', 3825914.37, 7925929.18],
['Divalproex Sodium', 2467067.02, 2708053.52],
['DIVALPROEX SODIUM', 1130014.83, 2604075],
['Divalproex Sodium Extended-Release', 720231.01, 1808978.69],
['Dorzolamide HCl', 95912.97, 708891.28],
['Double Antibiotic', 914881.77, 1958490.97],
['Dupixent', 1437439.99, 12654492.33],
['Eldepryl', 2128714.17, 3485649.76],
['EltaMD SPF 150 Sun Screen', 3389863.82, 3257976.38],
['Enalapril Maleate', 838168.71, 1342015.48],
['ENALAPRIL MALEATE', 2593528.67, 2564743.39],
['ESIKA', 569738.58, 1540350.32],
['eszopiclone', 458208.23, 681903.65],
['Famotidine', 1161623.36, 1358711.57],
['Flu-Cold', 907074.49, 1752274.54],
['FLU KIDS RELIEF', 772824.85, 1045454.54],
['Fumaderm', 195721.74, 2517600.95],
['Furosemide', 493523.99, 2738142.96],
['Gabapentin', 560320.28, 1094440.19],
['Gelato Topical Anesthetic', 2002106.22, 2429913],
['Glipizide', 261606.11, 3329242.48],
['Gold Bond Ultimate Healing Concentrate', 1242490.08, 2807831.05],
['Green Guard Stomach Relief', 1582876.55, 3560527.35],
['Haloperidol', 123085.04, 1995651.94],
['Hamamelis Virginiana Kit Refill', 365016.1, 586961.45],
['Hand Sanitizer with Moisturizers', 152614.97, 657085.12],
['Hand wash', 212607.07, 507628.5],
['Herceptin', 82371.13, 1428293.39],
['Humira', 3243809.46, 84759462.01],
['Hydralazine Hydrochloride', 250683.09, 1146201.6],
['Hydrochlorothiazide', 1201044.27, 1115255.32],
['Ibuprofen', 588699.33, 2313335.25],
['Ibuprofen PM', 410405.24, 3179533.5],
['Imatinib', 302721.56, 523311.9],
['Imraldi', 64948.76, 875514.67],
['IOPE RETIGEN MOISTURE FOUNDATION', 1397575.13, 1803199.39],
['Isoniazid', 1774390.9, 3379737.8],
['JUNIPERUS ASHEI POLLEN', 417476.74, 3581454.82],
['KADCYLA', 1096231.45, 2543749.63],
['Keflex', 999626.88, 1857681.77],
['Kentucky Bluegrass (June), Standardiz', 1519635.54, 2748508],
['Keytruda', 2137439.99, 13759462.01],
['Lamivudine and Zidovudine', 2974975.36, 2753546],
['Lamotrigine', 789582.98, 2292546.89],
['Lancome Paris Renergie Lift Volumetry', 1769907.97, 1825970],
['lansoprazole', 413382.39, 632705.44],
['LBel', 1573992.41, 1499768.09],
['Levetiracetam', 423335.39, 1780520.27],
['Levofloxacin', 921206.75, 949514.05],
['Levothyroxine Sodium', 2117864.7, 2937100],
['Lexapro', 1023275.76, 1220029.58],
['Lidocaine Hydrochloride and Epinephri', 508144.71, 566696],
['Listerine Ultraclean Antiseptic', 669228.79, 1879590.38],
['Locoid', 940593.68, 1084996.13],
['Losartan Potassium', 3804542.2, 3740527.69],
['Lovastatin', 2614556.03, 2996318.23],
['Lovenox', 796869.55, 3786377],
['Lucentis', 1649161.98, 2944223.22],
['Medi-Chord', 140422.87, 813188.82],
['Medi-First Cold Relief', 1083810.04, 3556698.88],
['Meloxicam', 2249571, 2524229],
['Methadone Hydrochloride', 2322161.57, 2582349],
['METHOCARBAMOL', 92756.59, 674397.16],
['Methylphenidate Hydrochloride', 321027.88, 821214.25],
['Metoclopramide', 1553229.42, 2341257.97],
['MiraLAX', 2898165.87, 3106931.54],
['Monistat Complete Care Instant Itch R', 240493.13, 904145],
['MooreBrand Ibuprofen', 344270, 1033654.77],
['Morphine Sulfate', 1058358.52, 3189843.38],
['Motion Sickness II', 1693474.52, 2786292.44],
['Motrin', 931084.25, 837620.18],
['Moxifloxacin Hydrochloride', 278350.32, 967632.61],
['MUCOR PLUMBEUS', 1177021.82, 3417877.93],
['Multi Symptom Cold', 757724.19, 1270441.66],
['Namenda', 1505831.15, 2510105],
['Naproxen Sodium', 82418.43, 1442991.59],
['Naratriptan', 679925.49, 3283823],
['Nefazodone Hydrochloride', 1298844.51, 2468436],
['Niacin', 1731552.31, 3208125],
['Nicorobin Clean and Clear', 3035522.06, 2930134.52],
['Night Time Cherry Syrup', 282038.76, 461623.76],
['Night-Time Original', 352858.53, 1485507.76],
['Nivolumab', 991099.63, 1658835.03],
['Non Aspirin PM', 1749148.23, 2204939.12],
['Nortriptyline Hydrochloride', 1485390.52, 3023426],
['N - TIME SINUS', 2414037.51, 2538701.5],
['Olay Total Effects Pore Minimizing CC', 1067854.28, 1415772.18],
['OLUX-E', 225171.93, 2443119.04],
['Orsythia', 848011.39, 1104640.13],
['Oxaprozin', 1439533.27, 1615518.35],
['Oxygen', 670011.51, 1443369.73],
['PDI Sani-Hands for Kids', 1598276.66, 1778024],
['Pepcid AC Acid Reducer', 2462370.76, 2104765],
['Persantine', 2020543.63, 2878612.16],
['Pfizerpen', 862377.49, 2896459.68],
['Potassium Chloride in Dextrose and So', 1105672.68, 1408872.02],
['Pramipexole Dihydrochloride', 1021908.39, 1112253.82],
['Pravastatin Sodium', 2095983.81, 2912543.88],
['Prednisone', 1021659.26, 1862729.02],
['ProBLEN Estrogen and Progesterone', 1206180.64, 2218026.75],
['PROMETHAZINE HYDROCHLORIDE', 553770.53, 961716.2],
['QUETIAPINE FUMARATE', 1511930.53, 2700252],
['RED GINSENG FERMENTED ESSENCE BB', 1285326.93, 1184664.57],
['RED ORANGE SUN', 565920.33, 1080726.88],
['Remicade', 134244.15, 1893117.08],
['RHIZOPUS ARRHIZUS VAR ARRHIZUS', 1584233.81, 2529937.41],
['Rimmel London', 694560.97, 2108286],
['Riociguat', 715430.59, 1253627.93],
['Ropinirole Hydrochloride', 2509926.79, 3091720.63],
['Sanitary Wipes Plus', 99811.26, 86938.27],
['Selegiline Hydrochloride', 1765245.63, 2723073.91],
['Sheep Sorrel Pollen', 992181.28, 3401782.44],
['SKIN FOUNDATION MINERAL MAKEUP', 1065324.19, 2938804.92],
['Sodium Iodide I 123', 887895.38, 1114438.97],
['Spot Repairing Serum', 1806344.97, 1755300.92],
['SPRYCEL', 387524.97, 1466145.44],
['Stavudine', 2069390.45, 3217507.49],
['Stay Awake', 1172798.04, 1528747.03],
['STEMPHYLIUM SOLANI', 1898929.25, 3358913.43],
['SunZone Work Sunscreen SPF-60', 1024897.45, 1727624.67],
['Surmontil', 521182.16, 600997.19],
['Synthroid', 2864248.62, 3147031.19],
['TAMSULOSIN HYDROCHLORIDE', 509536.85, 2843262.48],
['Thyroid Assist', 712265.84, 1379945.1],
['Tizanidine Hydrochloride', 263857.96, 1433109.5],
['Topcare Tussin', 1378790.53, 1304837.86],
['Tranexamic Acid', 2146503.01, 2752163],
['Treatment Set TS347116', 398395.47, 1743859.96],
['Treflan', 2522812.85, 2851411.66],
['Triple Complex Brain Tonic', 707891.52, 933692.06],
['UP and UP', 1373721.7, 2041758.41],
['Valproic Acid', 1698018.23, 2928590.94],
['Varicose Relief', 419174.97, 500101.61],
['Venlafaxine Hydrochloride', 640063.57, 697276.33],
['Wal-Zan', 779520.88, 723841.23],
['Warfarin Sodium', 316753.86, 822716.26],
['Western Family Pain Relieving', 961852.31, 1364046.09],
['Xanax', 7083504.56, 14759462.01],
['Xarelto', 192743.12, 985931],
['XtraCare Foaming Facial Cleanser', 156664.74, 3765829],
['Zavesca', 1072766.97, 2257132.08],
['Zyprexa', 208876.01, 293452.54],
['Zyrtec Ultra-Strength', 1313174.69, 1119479.36]
]

def convert_to_float(data):
    return [[item[0], float(item[1]), float(item[2])] for item in data]

_pa_data_float = convert_to_float(_pa_data)

df_pa_1 = spark.createDataFrame(_pa_data_float, _pa_1_schema)

df_pa_1_final = df_pa_1.groupBy(col("drugs"))\
       .agg(
         (sum(col("total_sales")) - sum(col("cogs"))).alias("profit")
       )\
      .orderBy(col("profit").desc())\
      .limit(3)\
      
df_pa_1_final.display()



_Pharmacy Analytics (Part 2)_

In [0]:
_pa_2_schema = StructType(
  [
    StructField("manufacturer", StringType()),
    StructField("drug", StringType()),
    StructField("cogs", DoubleType()),
    StructField("total_sales", DoubleType())
  ]
)

_pa2_data = [
  
['AbbVie','  Allopurinol' , 710573.44, 1425027.99],
['AbbVie','  Chlorzoxazone' , 595485.21, 1336647.6],
['AbbVie','  Citalopram' , 407210.29, 1201626.71],
['AbbVie','  Clarithromycin' , 3692136.66, 3499574.92],
['AbbVie','  Glipizide' , 261606.11, 3329242.48],
['AbbVie','  Hamamelis Virginiana Kit Refill' , 365016.1, 586961.45],
['AbbVie','  Humira' , 3243809.46, 84759462.01],
['AbbVie','  Hydralazine Hydrochloride' , 250683.09, 1146201.6],
['AbbVie','  Lamivudine and Zidovudine' , 2974975.36, 2753546],
['AbbVie','  Lexapro' , 1023275.76, 1220029.58],
['AbbVie','  Lidocaine Hydrochloride and Epinephri' , 508144.71, 566696],
['AbbVie','  Moxifloxacin Hydrochloride' , 278350.32, 967632.61],
['AbbVie','  Night-Time Original' , 352858.53, 1485507.76],
['AbbVie','  Olay Total Effects Pore Minimizing CC' , 1067854.28, 1415772.18],
['AbbVie','  Sodium Iodide I 123' , 887895.38, 1114438.97],
['AbbVie','  Stavudine' , 2069390.45, 3217507.49],
['AbbVie','  Valproic Acid' , 1698018.23, 2928590.94],
['AbbVie','  Warfarin Sodium' , 316753.86, 822716.26],
['AstraZeneca','  Armour Thyroid' , 956693.99, 1151498.6],
['AstraZeneca','  BANANA BOAT SUNSCREEN' , 163610.19, 937901.5],
['AstraZeneca','  Double Antibiotic' , 914881.77, 1958490.97],
['AstraZeneca','  Enalapril Maleate' , 838168.71, 1342015.48],
['AstraZeneca','  Furosemide' , 493523.99, 2738142.96],
['AstraZeneca','  Green Guard Stomach Relief' , 1582876.55, 3560527.35],
['AstraZeneca','  lansoprazole' , 413382.39, 632705.44],
['AstraZeneca','  Listerine Ultraclean Antiseptic' , 669228.79, 1879590.38],
['AstraZeneca','  MiraLAX' , 2898165.87, 3106931.54],
['AstraZeneca','  Naproxen Sodium' , 82418.43, 1442991.59],
['AstraZeneca','  PDI Sani-Hands for Kids' , 1598276.66, 1778024],
['AstraZeneca','  Pravastatin Sodium' , 2095983.81, 2912543.88],
['AstraZeneca','  SKIN FOUNDATION MINERAL MAKEUP' , 1065324.19, 2938804.92],
['AstraZeneca','  SPRYCEL' , 387524.97, 1466145.44],
['AstraZeneca','  Surmontil' , 521182.16, 600997.19],
['AstraZeneca','  Thyroid Assist' , 712265.84, 1379945.1],
['AstraZeneca','  Zavesca' , 1072766.97, 2257132.08],
['Bayer','  Acyclovir Sodium' , 616632.58, 841911.12],
['Bayer','  Amitriptyline Hydrochloride' , 2705112.11, 4701990.22],
['Bayer','  Citalopram Hydrobromide' , 412421.37, 1355861.1],
['Bayer','  Claritin' , 412654.26, 2825302],
['Bayer','  Diphenhydramine HCL' , 256621.14, 778332.7],
['Bayer','  ENALAPRIL MALEATE' , 2593528.67, 2564743.39],
['Bayer','  Gabapentin' , 560320.28, 1094440.19],
['Bayer','  Ibuprofen PM' , 410405.24, 3179533.5],
['Bayer','  Levofloxacin' , 921206.75, 949514.05],
['Bayer','  Lovastatin' , 2614556.03, 2996318.23],
['Bayer','  Meloxicam' , 2249571, 2524229],
['Bayer','  MooreBrand Ibuprofen' , 344270, 1033654.77],
['Bayer','  Oxygen' , 670011.51, 1443369.73],
['Bayer','  PROMETHAZINE HYDROCHLORIDE' , 553770.53, 961716.2],
['Bayer','  Riociguat' , 715430.59, 1253627.93],
['Bayer','  Sheep Sorrel Pollen' , 992181.28, 3401782.44],
['Bayer','  Triple Complex Brain Tonic' , 707891.52, 933692.06],
['Bayer','  Xarelto' , 192743.12, 985931],
['Biogen','  Active-Medicated specimen collection' , 1124982.66, 1460189.98],
['Biogen','  Acyclovir' , 3427421.73, 3130097],
['Biogen','  Alprazolam' , 2014858.59, 3080474.64],
['Biogen','  AVAPRO' , 2327541.25, 2792775.1],
['Biogen','  Brucella Remedy' , 221803.73, 700562.13],
['Biogen','  Burkhart' , 1006447.73, 1084258],
['Biogen','  Clotrimazole' , 2707620.02, 2717420.96],
['Biogen','  Cuvposa' , 934986.87, 2933877],
['Biogen','  DIPHENHYDRAMINE HYDROCHLORIDE' , 453989.65, 1825783.98],
['Biogen','  ESIKA' , 569738.58, 1540350.32],
['Biogen','  Fumaderm' , 195721.74, 2517600.95],
['Biogen','  Gelato Topical Anesthetic' , 2002106.22, 2429913],
['Biogen','  Haloperidol' , 123085.04, 1995651.94],
['Biogen','  Imraldi' , 64948.76, 875514.67],
['Biogen','  KADCYLA' , 1096231.45, 2543749.63],
['Biogen','  Lamotrigine' , 789582.98, 2292546.89],
['Biogen','  Lancome Paris Renergie Lift Volumetry' , 1769907.97, 1825970],
['Biogen','  Losartan Potassium' , 3804542.2, 3740527.69],
['Biogen','  Medi-Chord' , 140422.87, 813188.82],
['Biogen','  Monistat Complete Care Instant Itch R' , 240493.13, 904145],
['Biogen','  Namenda' , 1505831.15, 2510105],
['Biogen','  Nefazodone Hydrochloride' , 1298844.51, 2468436],
['Biogen','  Non Aspirin PM' , 1749148.23, 2204939.12],
['Biogen','  Nortriptyline Hydrochloride' , 1485390.52, 3023426],
['Biogen','  N - TIME SINUS' , 2414037.51, 2538701.5],
['Biogen','  Orsythia' , 848011.39, 1104640.13],
['Biogen','  Pramipexole Dihydrochloride' , 1021908.39, 1112253.82],
['Biogen','  Prednisone' , 1021659.26, 1862729.02],
['Biogen','  ProBLEN Estrogen and Progesterone' , 1206180.64, 2218026.75],
['Biogen','  QUETIAPINE FUMARATE' , 1511930.53, 2700252],
['Biogen','  RED ORANGE SUN' , 565920.33, 1080726.88],
['Biogen','  RHIZOPUS ARRHIZUS VAR ARRHIZUS' , 1584233.81, 2529937.41],
['Biogen','  UP and UP' , 1373721.7, 2041758.41],
['Biogen','  Varicose Relief' , 419174.97, 500101.61],
['Biogen','  Wal-Zan' , 779520.88, 723841.23],
['Eli Lilly','  Androgel' , 675055.5, 3025655],
['Eli Lilly','  Cefaclor' , 908662.72, 1793428.19],
['Eli Lilly','  Cialis' , 3189863.82, 9654492.33],
['Eli Lilly','  Clobetasol Propionate' , 1754344.77, 3303186.78],
['Eli Lilly','  Dermasorb TA Complete Kit' , 2742445.9, 2521023.73],
['Eli Lilly','  Diclofenac Sodium' , 482212.61, 1545347.97],
['Eli Lilly','  Divalproex Sodium' , 2467067.02, 2708053.52],
['Eli Lilly','  DIVALPROEX SODIUM' , 1130014.83, 2604075],
['Eli Lilly','  Eldepryl' , 2128714.17, 3485649.76],
['Eli Lilly','  IOPE RETIGEN MOISTURE FOUNDATION' , 1397575.13, 1803199.39],
['Eli Lilly','  Keflex' , 999626.88, 1857681.77],
['Eli Lilly','  Kentucky Bluegrass (June), Standardiz' , 1519635.54, 2748508],
['Eli Lilly','  LBel' , 1573992.41, 1499768.09],
['Eli Lilly','  Levetiracetam' , 423335.39, 1780520.27],
['Eli Lilly','  Levothyroxine Sodium' , 2117864.7, 2937100],
['Eli Lilly','  Metoclopramide' , 1553229.42, 2341257.97],
['Eli Lilly','  Morphine Sulfate' , 1058358.52, 3189843.38],
['Eli Lilly','  Multi Symptom Cold' , 757724.19, 1270441.66],
['Eli Lilly','  Naratriptan' , 679925.49, 3283823],
['Eli Lilly','  Night Time Cherry Syrup' , 282038.76, 461623.76],
['Eli Lilly','  OLUX-E' , 225171.93, 2443119.04],
['Eli Lilly','  Pfizerpen' , 862377.49, 2896459.68],
['Eli Lilly','  Potassium Chloride in Dextrose and So' , 1105672.68, 1408872.02],
['Eli Lilly','  RED GINSENG FERMENTED ESSENCE BB' , 1285326.93, 1184664.57],
['Eli Lilly','  Rimmel London' , 694560.97, 2108286],
['Eli Lilly','  Spot Repairing Serum' , 1806344.97, 1755300.92],
['Eli Lilly','  STEMPHYLIUM SOLANI' , 1898929.25, 3358913.43],
['Eli Lilly','  Tizanidine Hydrochloride' , 263857.96, 1433109.5],
['Eli Lilly','  Treatment Set TS347116' , 398395.47, 1743859.96],
['Eli Lilly','  Treflan' , 2522812.85, 2851411.66],
['Eli Lilly','  Western Family Pain Relieving' , 961852.31, 1364046.09],
['Eli Lilly','  Zyprexa' , 208876.01, 293452.54],
['GlaxoSmithKline','  eszopiclone' , 458208.23, 681903.65],
['GlaxoSmithKline','  Isoniazid' , 1774390.9, 3379737.8],
['Johnson & Johnson','  Budpak Petroleum Jelly' , 107902.51, 1406533.69],
['Johnson & Johnson','  Childrens Ibuprofen' , 384241.13, 2835232.18],
['Johnson & Johnson','  Common Sagebrush' , 2755697.86, 3059122.36],
['Johnson & Johnson','  EltaMD SPF 150 Sun Screen' , 3389863.82, 3257976.38],
['Johnson & Johnson','  Flu-Cold' , 907074.49, 1752274.54],
['Johnson & Johnson','  Gold Bond Ultimate Healing Concentrate' , 1242490.08, 2807831.05],
['Johnson & Johnson','  Hand Sanitizer with Moisturizers' , 152614.97, 657085.12],
['Johnson & Johnson','  Hand wash' , 212607.07, 507628.5],
['Johnson & Johnson','  Ibuprofen' , 588699.33, 2313335.25],
['Johnson & Johnson','  JUNIPERUS ASHEI POLLEN' , 417476.74, 3581454.82],
['Johnson & Johnson','  Medi-First Cold Relief' , 1083810.04, 3556698.88],
['Johnson & Johnson','  Motrin' , 931084.25, 837620.18],
['Johnson & Johnson','  Niacin' , 1731552.31, 3208125],
['Johnson & Johnson','  Nicorobin Clean and Clear' , 3035522.06, 2930134.52],
['Johnson & Johnson','  Pepcid AC Acid Reducer' , 2462370.76, 2104765],
['Johnson & Johnson','  Remicade' , 134244.15, 1893117.08],
['Johnson & Johnson','  Sanitary Wipes Plus' , 99811.26, 86938.27],
['Johnson & Johnson','  SunZone Work Sunscreen SPF-60' , 1024897.45, 1727624.67],
['Johnson & Johnson','  XtraCare Foaming Facial Cleanser' , 156664.74, 3765829],
['Johnson & Johnson','  Zyrtec Ultra-Strength' , 1313174.69, 1119479.36],
['Merck','  DHEA' , 1233306.52, 1559816],
['Merck','  Divalproex sodium' , 3825914.37, 7925929.18],
['Merck','  FLU KIDS RELIEF' , 772824.85, 1045454.54],
['Merck','  Keytruda' , 2137439.99, 13759462.01],
['Merck','  METHOCARBAMOL' , 92756.59, 674397.16],
['Novartis','  Alimta' , 575583.41, 1194250.01],
['Novartis','  Amoxicillin and Clavulanate Potassium' , 981993.64, 1518343],
['Novartis','  ANASTROZOLE' , 2560951.38, 3116140.22],
['Novartis','  Antiseptic Hand Gel' , 436399.12, 1734572],
['Novartis','  Diltiazem Hydrochloride' , 1041602.54, 1796908.48],
['Novartis','  Famotidine' , 1161623.36, 1358711.57],
['Novartis','  Imatinib' , 302721.56, 523311.9],
['Novartis','  Xanax' , 7083504.56, 14759462.01],
['Pfizer','  Ciprofloxacin' , 1278017.17, 2115658],
['Pfizer','  clobetasol propionate' , 1281909.78, 2582356],
['Pfizer','  Diaper Rash Skin Protectant Crema Cer' , 426539.59, 537795],
['Pfizer','  Divalproex Sodium Extended-Release' , 720231.01, 1808978.69],
['Pfizer','  Methadone Hydrochloride' , 2322161.57, 2582349],
['Pfizer','  Methylphenidate Hydrochloride' , 321027.88, 821214.25],
['Pfizer','  MUCOR PLUMBEUS' , 1177021.82, 3417877.93],
['Pfizer','  Ropinirole Hydrochloride' , 2509926.79, 3091720.63],
['Pfizer','  Selegiline Hydrochloride' , 1765245.63, 2723073.91],
['Pfizer','  Stay Awake' , 1172798.04, 1528747.03],
['Pfizer','  TAMSULOSIN HYDROCHLORIDE' , 509536.85, 2843262.48],
['Pfizer','  Tranexamic Acid' , 2146503.01, 2752163],
['Pfizer','  Venlafaxine Hydrochloride' , 640063.57, 697276.33],
['Roche','  Cloves' , 1202177.41, 1892705.42],
['Roche','  Crab' , 1164637.24, 1698544.77],
['Roche','  Dorzolamide HCl' , 95912.97, 708891.28],
['Roche','  Herceptin' , 82371.13, 1428293.39],
['Roche','  Hydrochlorothiazide' , 1201044.27, 1115255.32],
['Roche','  Lucentis' , 1649161.98, 2944223.22],
['Roche','  Motion Sickness II' , 1693474.52, 2786292.44],
['Roche','  Nivolumab' , 991099.63, 1658835.03],
['Roche','  Topcare Tussin' , 1378790.53, 1304837.86],
['Sanofi','  Dupixent' , 1437439.99, 12654492.33],
['Sanofi','  Locoid' , 940593.68, 1084996.13],
['Sanofi','  Lovenox' , 796869.55, 3786377],
['Sanofi','  Oxaprozin' , 1439533.27, 1615518.35],
['Sanofi','  Persantine' , 2020543.63, 2878612.16],
['Sanofi','  Synthroid' , 2864248.62, 3147031.19]
]

def convert(data):
  return [ [item[0],item[1], float(item[2]),float(item[3])] for item in data]

final_data = convert(_pa2_data)

df_pa_2 = spark.createDataFrame(final_data, _pa_2_schema)

df_pa_2.where(col("cogs") > col("total_sales"))\
      .groupBy(col("manufacturer"))\
       .agg(
         count(col("drug")).alias("drug_count"),
         sum(col("cogs") - col("total_sales")).alias("total_loss")
       )\
      .orderBy(col("total_loss").desc())\
      .limit(6)\
      .display()


_Patient Support Analysis (Part 1)_

In [0]:
_data_psa_schema = StructType([
  StructField("p_id", IntegerType()),
  StructField("c_id", StringType()),
])

_data_psa_1 = [
[1 ,'f1d012f9-9d02-4966-a968-bf6c5bc9a9fe'],
[1 ,'41ce8fb6-1ddd-4f50-ac31-07bfcce6aaab'],
[2 ,'8471a3d4-6fc7-4bb2-9fc7-4583e3638a9e'],
[2 ,'38208fae-bad0-49bf-99aa-7842ba2e37bc'],
[3 ,'f0e7a8e3-df93-40f3-9b5e-fadff9ebe072'],
[3 ,'b72f91e6-c3f8-4358-a1f2-c9507e8dcba4'],
[3 ,'3acbe22d-22b3-4144-954d-74c127bc49ea'],
[3 ,'e32b61c2-a90d-4371-a5ee-6bc44fa49bbd'],
[3 ,'6099f469-b5d6-4447-9acf-d936355eae7c'],
[4 ,'1c6896df-ddfb-48c2-ade1-f9fb2da2fc53'],
[5 ,'5f5d013d-b95f-4e9c-9817-ca84ae537566'],
[5 ,'96252f76-7a41-484d-ae75-a2ae58e1797c'],
[5 ,'21cfd997-40e7-461b-b5ac-b29b559b216a'],
[6 ,'aef712ee-e95f-47d3-93df-642ddb914763'],
[7 ,'5bfadc8a-7e00-458c-a45d-6b424054a61e'],
[7 ,'301bb345-c66c-4547-81a5-9d9ab869765c'],
[7 ,'b648cc45-b457-40aa-8125-e13daeddf9fb'],
[7 ,'d3fc2c77-7643-4ac3-9ee9-ded5a3bc7c5e'],
[7 ,'a4d928b9-79ad-4fd2-9468-108b98ca1843'],
[8 ,'5d2227eb-f208-4923-8591-8d4939a4381a'],
[8 ,'e85346a4-44ad-4826-be26-0260de338db6'],
[8 ,'a2571e64-eaac-495c-aced-a6e9b537991e'],
[9 ,'310f2276-776a-4d31-9da6-80dd77879cce'],
[9 ,'81d7b924-464e-4cf2-ad97-f2dc7d862ff3'],
[10 ,'d049140c-fc7c-4674-8bb1-fc642a8828e6'],
[10 ,'ad5d604a-bb45-4914-ab2b-fe3136327f26'],
[10 ,'b6a19973-c47c-46c2-9b29-7c7e3c812d6d'],
[10 ,'3270aed0-87d0-4a38-82ba-361e61299481'],
[11 ,'ae3458fe-d75f-41c6-bfdc-0ac6f76b1074'],
[11 ,'10313c16-5a86-4b94-9ee1-afece6248ad2'],
[11 ,'85cdf7c5-a665-4e93-8d42-097d4692794f'],
[11 ,'aaee907f-2a5f-4dae-8dfb-9875bb660b5e'],
[11 ,'ddb05617-b459-4234-aba3-e48701410524'],
[11 ,'139f011b-2844-4879-96c6-cb51e09cffcc'],
[11 ,'d0543b64-8494-4c5a-b749-705eca1a1302'],
[11 ,'ace4fa7f-90f3-4fdb-a85e-16c672930123'],
[12 ,'44f00f9d-5d72-4833-a6fd-e4cd27cd2d15'],
[12 ,'b4a2c2f2-718a-48a3-b9e1-90acb5d233f7'],
[12 ,'a23716c5-faae-49a2-a0b4-c99fb8fd461a'],
[12 ,'8f6bb7ee-7672-4d76-a14f-71906e76d6b6'],
[13 ,'894a08c6-82c6-49ed-84a0-b627752cdca7'],
[13 ,'7ee4e5f0-4924-47f7-8809-172bb59c6ef0'],
[13 ,'16267196-57e3-436e-8100-2563f2c71791'],
[14 ,'090c3dad-c756-4dec-a313-b9721d4f93fc'],
[14 ,'0b68aafa-6907-43c3-9dbd-a6c171cf5006'],
[14 ,'c4660294-9443-4aba-b6d4-f39a9d5e5f00'],
[14 ,'d357ff9e-baa2-4361-9dd4-ceeb318b14c0'],
[14 ,'ddf5c58c-e439-4e00-be3b-788575ac4f02'],
[14 ,'902a5c82-c0f5-4f5e-bee9-db4fe40523a2'],
[15 ,'06545ac5-18f5-4ae8-9b1e-087c7dc8deca'],
[15 ,'9580a1ad-842a-482f-a05c-7e1c09b926b3'],
[15 ,'113632a8-27ac-46aa-a881-015f3554b21c'],
[15 ,'c525f468-b467-4947-bf22-29fba02397a7'],
[15 ,'cafe928e-e68b-4883-840e-7f79f1fb24f9'],
[15 ,'336588e7-1856-49f9-b91a-6a14e664c30b'],
[15 ,'7d1525b6-b58a-41cd-9a22-67259889e0d9'],
[16 ,'16cb7643-1708-4d5b-8d9d-dbc9fb411045'],
[16 ,'bc6b946b-6587-474e-984a-7ec972437f59'],
[16 ,'915ea199-f505-44c9-b3ff-6ccc6981c166'],
[17 ,'f6cef3fd-4c78-49d9-b243-6b4753240744'],
[17 ,'fea48e73-25ea-419c-be69-1a62ebc8ebcc'],
[17 ,'18fcdc02-6305-4f75-b55e-ddf9b0809133'],
[18 ,'6c835168-19ca-4d1a-9a4c-a0b608d10f2d'],
[18 ,'0553c2b2-b156-4f0d-8881-259974a219b9'],
[19 ,'58d5deb8-7e1f-4118-a3ac-6e40c1b8f4ac'],
[19 ,'4c574f95-cdc1-43e9-8963-876491bd173c'],
[19 ,'4e029697-2ea0-4791-9894-e20aecf29930'],
[19 ,'179281a0-028c-419c-afa9-fa64db8973de'],
[20 ,'f020c8c1-5653-49f7-9860-0075c8cf2643'],
[20 ,'17011401-a02a-4322-85ee-df04dac5c524'],
[3 ,'2e396f53-9b43-4dd5-9ed3-691bf1a191ce'],
[8 ,'0528a373-4093-4619-abc0-1da7512d87c0'],
[15 ,'b8ddc823-e855-4ab6-92a9-1b40b0d7545d'],
[19 ,'2455cdc9-ae47-44fe-8747-0f2512904920'],
[20 ,'8a5b020b-8d6c-44a4-beb7-3db8e1b363bd'],
[20 ,'79aeda21-78e6-4430-9f80-ea23f0222f22'],
[21 ,'0277e8a0-1666-4058-8b6d-e7a6e623a1f8'],
[21 ,'ee8cb1b7-9b48-4675-929a-fa095f0a9ec6'],
[21 ,'bfac6c1c-be86-4a5f-9548-f8d1e9143c3d'],
[21 ,'a9ff6c7e-2f75-4fc2-b6bd-b7c963fb3864'],
[21 ,'dfc113a5-b399-4543-be25-7a79e543897a'],
[22 ,'df184ff9-de31-4d2f-8041-b1b5f0ce7bc2'],
[22 ,'0fa2685b-454a-4296-8e7c-2029b60a0bd6'],
[23 ,'adfa3fc0-4eb6-4439-ab53-58e844d86e3d'],
[23 ,'24947dc8-b656-495a-9ced-8de94260219b'],
[23 ,'9fc973de-6d66-490f-865e-39f8f3d755be'],
[23 ,'8a1645bf-7d5d-467c-9734-4f78c5e24a3e'],
[24 ,'6facf446-f988-41a1-8615-7b418df13770'],
[24 ,'12bfa9d5-2cb7-4cb2-9529-e96f29d6e289'],
[25 ,'6944ed06-f1db-4ce0-acbf-199a8be5b41a'],
[25 ,'6432857e-e801-4116-8c08-79ebfcb10f8c'],
[26 ,'386e4b5d-c038-40da-a938-e3b397a6ee00'],
[26 ,'be5bcdd8-8e2a-4ae2-97ac-bd69aef57a36'],
[26 ,'9d688114-5a23-4bdf-a198-9ded68c1569e'],
[26 ,'fd15fcf5-954f-40f6-b2c3-673315779446'],
[27 ,'39e427b3-4347-475d-8e5a-a830cd8b4d42'],
[27 ,'70fad67f-a08c-4300-acf3-5799538627bb'],
[27 ,'2a776ddb-1712-4f24-bf94-dbb86591df9a'],
[27 ,'ca4030d9-04a9-4f3d-a92b-d857f4f77ac3'],
[27 ,'1c227359-43a9-481e-bbcb-5840696f8c53'],
[27 ,'5f486e61-7cad-4d1f-b297-5aace135499c'],
[28 ,'28fc7d84-dc9c-4d99-b0f2-06e910a0fc9e'],
[28 ,'c46be667-b94e-40bf-bb97-ba5da51bc8e8'],
[28 ,'def1a0a1-bd87-465e-8df5-2baa2359b4f8'],
[28 ,'ff71730f-cefe-499e-a167-1cd346e0589e'],
[28 ,'ee827e54-0ddb-4ea1-ac49-f0d25e69f488'],
[28 ,'270e4724-3a64-4e34-a4fa-570fc27004aa'],
[28 ,'74d68088-7776-42de-ac5b-296da74fbcf9'],
[28 ,'b1f71139-d23b-48ba-ae00-66cc600eb14c'],
[28 ,'002a7deb-8b80-40af-be55-5d7c4c827450'],
[29 ,'37f3a927-2c74-4de1-a9fc-2cdd0d1380c8'],
[29 ,'0df44526-8eac-4090-ac2f-2f1ce3deec89'],
[29 ,'a951a929-4ba6-455b-9024-bf61af77d8ee'],
[30 ,'2e7fce0b-9eda-46d0-a932-156821f117f7'],
[31 ,'9d80df96-8cf0-4749-aed9-4d92fd36ca81'],
[31 ,'463fd8f6-2d3b-4493-8c39-a4b45e521911'],
[32 ,'a3241817-d541-4e38-8e99-74e28980eca3'],
[32 ,'1502de8a-0398-43f5-8088-30c823df27a3'],
[32 ,'94ff196e-4405-4ccf-b41f-68eb007d1f4c'],
[32 ,'26b2e19a-d6d5-4874-92af-a986aeaa36b7'],
[32 ,'562af373-0c56-4a32-8a3a-6fd7901c0b65'],
[32 ,'63614152-ad1c-4e03-b747-94235cfb2c64'],
[32 ,'04d35acc-8a45-4374-a1aa-a25865cec3ab'],
[33 ,'7f7a5ad3-192c-4fbf-a008-094eef5ee074'],
[33 ,'dcbbb1ec-7278-4bc0-b8d4-ce308125f997'],
[33 ,'53869322-9c84-41da-bf0c-7425676fba5b'],
[33 ,'e360a497-dcd2-4697-8f6b-16882d4fe6df'],
[34 ,'19858846-4281-4e1c-b2d7-988b0419dac2'],
[34 ,'09fe32e9-c926-4911-b305-916378eff145'],
[34 ,'3817e7bf-d49d-48d3-adf9-4802aea82f86'],
[34 ,'b1283ebd-fa27-4275-ac9c-014de62fbd3e'],
[35 ,'35e54352-0739-42fe-b322-ab9074b0b6a3'],
[36 ,'1bebdd5b-9ba9-4e0a-8960-6af0ca12e592'],
[36 ,'5f3b4140-11ae-4c99-ba31-be97c9a3440b'],
[36 ,'427bf9d3-4688-4d42-8280-dcaf2c7e1031'],
[36 ,'de523a4d-7941-4395-84f8-6bf01639d626'],
[36 ,'1aa8b357-6511-4fb8-b38e-81621449de8a'],
[36 ,'e38b63a7-a8a8-4661-9906-1519510c1246'],
[36 ,'5b689d8f-bb83-48e0-9fba-c3e507673939'],
[36 ,'6ef57650-1fc7-4a33-ab97-72b7e7828ac2'],
[36 ,'69f5409e-9bb7-4fbf-8512-9a75f9281fe3'],
[37 ,'f3ca890b-3f78-415e-b85c-6cb2c3aaab51'],
[37 ,'c94f7c79-b003-4859-ae36-d049441b65ce'],
[37 ,'db4baddb-b6a9-4f8a-8e40-fd7c98b2b514'],
[37 ,'c1df30f6-7d1b-4420-9cd6-64dc73e14afe'],
[37 ,'e71f022a-83e3-47ad-b2ce-b69429649e4c'],
[37 ,'10000157-602f-47f6-b759-c3cbef6c8b13'],
[23 ,'693a2e20-6b77-43df-90f3-28698921027e'],
[38 ,'422d6539-4bbb-456d-8f79-587ecbd55e30'],
[38 ,'0445d36f-48a7-4042-bb29-efa2a48ea630'],
[38 ,'08d55cb0-298d-4a3c-b290-e4f4e7a374c1'],
[38 ,'a1c95253-eca7-49bb-8b66-7e4e8ec35558'],
[38 ,'fb873afd-7134-45ed-bd27-4a522dd8623f'],
[39 ,'af005577-b2a8-43e5-9c70-835f1e566971'],
[39 ,'d9667b64-b888-4c1e-8e91-556aca534f9e'],
[39 ,'c29d2a60-003e-4666-9857-243d2a13b817'],
[40 ,'25fd6b89-5334-4716-bd25-806e0239998f'],
[40 ,'85e89309-14e6-47e1-baf3-9168639b2f0e'],
[40 ,'e1876a2e-56da-42e5-b99c-fa47ce9151bd'],
[40 ,'0cb29627-feea-43e2-ade3-ef1784748392'],
[41 ,'c42daca5-50a2-45c1-acf1-1bc5ce3fa0bf'],
[41 ,'4e83365e-0da1-4c7d-af47-1992ed38cba0'],
[42 ,'c84c816d-bebc-4738-8a23-4db03454eb0b'],
[42 ,'650e9242-3d20-4ce4-948f-d1a896344aa9'],
[42 ,'e378005e-4174-44e7-b511-712a2e8b2e5f'],
[42 ,'d57bb587-3676-4b5f-8d0d-4fbd0e386039'],
[42 ,'5010f181-320c-49c8-ba76-4b89c6433778'],
[43 ,'8437cb2e-9395-4d94-acde-bc154bdcb1d1'],
[43 ,'80ed01fc-edbe-46e0-a558-3da49305a70d'],
[43 ,'183f4d60-a472-4603-9301-c66936cfa332'],
[44 ,'10b9dda5-550f-4de6-88d7-793a5c8a7498'],
[44 ,'184ed80c-a6c8-4483-9304-39cd8c4cacd2'],
[45 ,'36a92072-1f6c-4fd1-a0c5-63857abb72c7'],
[45 ,'6074ddd0-db0a-40a6-9771-4ea46461cc72'],
[45 ,'4f7d72ab-9477-46f1-9da0-eeb7296ecd90'],
[45 ,'37c5e11e-91b8-484f-8b63-acff6c8b8ea4'],
[45 ,'2aae3e64-206e-43f5-8230-80f24ea2cdb1'],
[45 ,'d34bc623-739b-4fc1-8f7d-44b4c53ba4ec'],
[46 ,'efd308bc-9f26-44c2-a2d1-6ca50c145216'],
[46 ,'7d698dab-45f4-4b30-8ed9-60f6e78c9166'],
[46 ,'36d7e995-8dda-44fd-9cad-4f314fb34a93'],
[46 ,'e582717e-fa6d-4ca2-8fe9-e754340c3b5f'],
[46 ,'8ac501ef-6447-4e54-a8ce-706ef2feca42'],
[46 ,'d254fc36-e740-4e68-90de-33d67a2caaea'],
[47 ,'b1151347-8ecf-4cf4-bfae-fc4bff20ee54'],
[47 ,'22460ba8-179e-4be7-b108-a06afcf027b1'],
[47 ,'b2da6801-4037-42cf-a321-ab44d22f61dd'],
[47 ,'9e31ca0c-d36c-4a36-a9fe-336a304e0ac7'],
[48 ,'6cd0998d-e142-4e60-b043-f93637483997'],
[48 ,'ebacd999-097c-4f34-a80e-4818f7d6d675'],
[48 ,'7b23882d-f0d2-4ee7-91bd-96c8d7147ef0'],
[49 ,'86592103-61ec-4d81-9fd5-223875a6158d'],
[50 ,'6e3473ca-7c3a-493b-89ac-07aa308e2b92'],
[50 ,'2777e878-dbca-4834-b7e4-62e72b2938be'],
[2 ,'9b1af84b-eedb-4c21-9730-6f099cc2cc5e'],
[22 ,'31c10d2b-d0af-4909-b4e8-7cc676afe763'],
[38 ,'32240b3b-dfac-460f-9378-a7e7203ac6b8'],
[41 ,'1811ffd7-6fd1-4849-a6d7-87b41ab96d78'],
[49 ,'2b0c0ff7-36f3-4c06-81a0-6f79984a9fe2']
]

df_psa_1 = spark.createDataFrame(_data_psa_1 ,_data_psa_schema)

df_psa_2 = df_psa_1.groupBy("p_id")\
        .agg(
          countDistinct(col("c_id")).alias("c_id_cnt")
        )\
        .where(col("c_id_cnt") >= 3)\
        .select(col("p_id"))\

df_psa_2.count()